###  Code Agent

agent本质是对https请求的精致封装

### Cell 1 · 先发一次最简单的基础对话

看一次最小的 Chat Completions 调用

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from openai import OpenAI
model = os.getenv("DEEPSEEK_MODEL", "deepseek-flash")
client = OpenAI(api_key=os.environ.get("DEEPSEEK_API_KEY"), base_url=os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com"))
basic = client.chat.completions.create(model=model, messages=[{"role": "user", "content": "你好"}])
dump = lambda value: print(json.dumps(value.model_dump(exclude_none=True), ensure_ascii=False, indent=2))
dump(basic)

### Cell 2 · 返回值字段拆开

上一格拿到的是一个带有请求 id、模型名、choices、message、finish reason、usage 的对象

In [ ]:
response_data = basic.model_dump()
print("top-level fields:", list(response_data))
print("choices fields:", list(response_data["choices"][0]))
print("message fields:", list(response_data["choices"][0]["message"]))
print("usage fields:", list(response_data.get("usage", {})))
print("full response:")
print(json.dumps(response_data, ensure_ascii=False, indent=2))

### Cell 3 · 模型本质只接收一个 message 列表

Agent 的记忆不是模型内部自动保存的东西，而是下一次请求继续把旧消息放进 `messages`。先放系统提示词，再依次添加用户、助手和下一条用户消息。系统提示词明确工作目录，所以后面的问题可以只写相对路径。

In [ ]:
WORKDIR = Path.cwd()
system_prompt = f"""你是本地软件工程代码助手。
工作目录是 {WORKDIR}。
用户要求读取、写入或运行本地代码时，使用本请求声明的工具，并把工具结果放回下一轮 messages。
回答要基于真实文件和真实执行结果。"""
messages = [{"role": "system", "content": system_prompt}]
messages.append({"role": "user", "content": "你好"})
messages.append({"role": "assistant", "content": basic.choices[0].message.content})
messages.append({"role": "user", "content": "接下来我们围绕当前工作目录继续。"})
print(json.dumps(messages, ensure_ascii=False, indent=2))

### Cell 4 · 没有工具时，模型只能给出说明

现在直接问一个本地文件问题。System Prompt 里虽然有工作目录，但请求没有工具契约，模型不知道该怎样读取文件，也没有规范的工具调用返回值，只能回答一段文字

In [ ]:
messages4 = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "请读取 evaluator.py，告诉我当前 evaluate 函数是怎么实现的。"},
]
reply4 = client.chat.completions.create(model=model, messages=messages4)
print(reply4.choices[0].message.content)

### Cell 5 · 加入 read_file 工具

工具封装的关键只有三步：声明 JSON Schema；把模型发来的参数交给本地函数；把函数结果作为 `role=tool` 放回消息列表，然后再请求一次模型。

In [ ]:
def read_file(path):
    return Path(path).read_text(encoding="utf-8")

read_file_tool = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "读取工作目录中的完整文件内容",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]},
    },
}
messages5 = [{"role": "system", "content": system_prompt}, {"role": "user", "content": "请读取 evaluator.py，完整告诉我它现在做什么。"}]
reply5 = client.chat.completions.create(model=model, messages=messages5, tools=[read_file_tool], tool_choice={"type": "function", "function": {"name": "read_file"}})
assistant5 = reply5.choices[0].message.model_dump(exclude_none=True)
messages5.append(assistant5)
call5 = assistant5["tool_calls"][0]
args5 = json.loads(call5["function"]["arguments"])
messages5.append({"role": "tool", "tool_call_id": call5["id"], "content": read_file(args5["path"])})
reply5_final = client.chat.completions.create(model=model, messages=messages5, tools=[read_file_tool], tool_choice="none")
messages5.append(reply5_final.choices[0].message.model_dump(exclude_none=True))
print(json.dumps(messages5, ensure_ascii=False, indent=2))

### Cell 6 · 进一步加入 write_file 工具

读到真实源码后，下一步是让模型提交完整的新文件。写工具仍然只是一个函数和一份 Schema，真正的状态推进依旧靠 assistant tool call → 本地执行 → tool result → 下一轮请求

In [ ]:
def write_file(path, content):
    target = Path(path)
    target.write_text(content, encoding="utf-8")
    return {"path": str(target), "characters": len(content), "status": "written"}

write_file_tool = {
    "type": "function",
    "function": {
        "name": "write_file",
        "description": "把完整文本写入工作目录中的文件",
        "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]},
    },
}
messages6 = [{"role": "system", "content": system_prompt}, {"role": "user", "content": "请把一个支持加减乘除和括号的 evaluate 实现写入 evaluator_demo.py。"}]
reply6 = client.chat.completions.create(model=model, messages=messages6, tools=[write_file_tool], tool_choice={"type": "function", "function": {"name": "write_file"}})
assistant6 = reply6.choices[0].message.model_dump(exclude_none=True)
messages6.append(assistant6)
call6 = assistant6["tool_calls"][0]
args6 = json.loads(call6["function"]["arguments"])
write_result6 = write_file(args6["path"], args6["content"])
messages6.append({"role": "tool", "tool_call_id": call6["id"], "content": json.dumps(write_result6, ensure_ascii=False)})
reply6_final = client.chat.completions.create(model=model, messages=messages6, tools=[write_file_tool], tool_choice="none")
messages6.append(reply6_final.choices[0].message.model_dump(exclude_none=True))
print(json.dumps(messages6, ensure_ascii=False, indent=2))

### Cell 7 · 还要有 run_python，写完必须执行

写入只说明文件落盘，不能说明逻辑正确。运行工具把测试代码交给独立 Python 子进程，并把退出码、标准输出和标准错误作为下一轮的依据

In [ ]:
def run_python(code):
    result = subprocess.run([sys.executable, "-c", code], cwd=Path.cwd(), capture_output=True, text=True)
    return {"returncode": result.returncode, "stdout": result.stdout, "stderr": result.stderr}

run_python_tool = {
    "type": "function",
    "function": {
        "name": "run_python",
        "description": "在工作目录中运行 Python 代码并返回完整执行结果",
        "parameters": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]},
    },
}
messages7 = [{"role": "system", "content": system_prompt}, {"role": "user", "content": "请运行断言：from evaluator_demo import evaluate; assert evaluate('2*(3+4)-5/5') == 13"}]
reply7 = client.chat.completions.create(model=model, messages=messages7, tools=[run_python_tool], tool_choice={"type": "function", "function": {"name": "run_python"}})
assistant7 = reply7.choices[0].message.model_dump(exclude_none=True)
messages7.append(assistant7)
call7 = assistant7["tool_calls"][0]
args7 = json.loads(call7["function"]["arguments"])
messages7.append({"role": "tool", "tool_call_id": call7["id"], "content": json.dumps(run_python(args7["code"]), ensure_ascii=False)})
reply7_final = client.chat.completions.create(model=model, messages=messages7, tools=[run_python_tool], tool_choice="none")
messages7.append(reply7_final.choices[0].message.model_dump(exclude_none=True))
print(json.dumps(messages7, ensure_ascii=False, indent=2))

### Cell 8 · 代码库太乱：把 build_graph 和 query_graph 加进来

到这里已经是一个能读、能写、能运行的代码 Agent。面对多文件代码库，继续把全文塞进 messages 会很快失控，所以再加一层图谱工具：先把文件、符号和关系建成图，再按问题查询局部证据。这部分沿用 `SE-H1` 里的 `CodeGraph`。

In [ ]:
# notebook/ sits inside the repo; the graph module and the repo root are one
# level up from the working directory this notebook is run in.
REPO = Path.cwd().parent.resolve()
sys.path.insert(0, str(REPO))
from code_graph import CodeGraph

graph = CodeGraph(REPO)
def build_graph():
    return graph.build()
def query_graph(query, limit=12):
    return graph.query(query, limit)

graph_tools = [
    {"type": "function", "function": {"name": "build_graph", "description": "建立或刷新本地代码图谱", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "query_graph", "description": "查询文件、符号和关系", "parameters": {"type": "object", "properties": {"query": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["query"]}}},
]
messages8 = [{"role": "system", "content": system_prompt}, {"role": "user", "content": "先建立 SE-H1 图谱，再查询 CodeAgent、CodeGraph 和 run_python 的关系。"}]
reply8 = client.chat.completions.create(model=model, messages=messages8, tools=graph_tools, tool_choice="auto")
assistant8 = reply8.choices[0].message.model_dump(exclude_none=True)
messages8.append(assistant8)
for call8 in assistant8.get("tool_calls", []):
    name8 = call8["function"]["name"]
    args8 = json.loads(call8["function"]["arguments"] or "{}")
    result8 = build_graph() if name8 == "build_graph" else query_graph(**args8)
    messages8.append({"role": "tool", "tool_call_id": call8["id"], "content": json.dumps(result8, ensure_ascii=False, indent=2)})
reply8_final = client.chat.completions.create(model=model, messages=messages8, tools=graph_tools, tool_choice="none")
messages8.append(reply8_final.choices[0].message.model_dump(exclude_none=True))
print(json.dumps(graph_tools, ensure_ascii=False, indent=2))
print(json.dumps(messages8, ensure_ascii=False, indent=2))

### Cell 9 · 关于compact

读代码库会产生很长的工具结果。compact 的最小机制很简单：保留稳定的 system 前缀，把旧的 user、assistant、tool 消息换成结构化摘要，再把新问题接在摘要后面。

In [ ]:
overloaded_messages = messages8 + [{"role": "tool", "tool_call_id": f"old-{index}", "content": json.dumps(graph.graph, ensure_ascii=False)} for index in range(4)]
summary = {
    "goal": "审查 SE-H1 代码库的 Agent、图谱和工具边界",
    "completed": ["build_graph 已执行", "query_graph 已返回 CodeAgent、CodeGraph 和 run_python 的关系"],
    "evidence": ["messages8 中的完整工具调用和结果", "SE-H1/.code-graph/graph.json"],
    "next_step": "继续处理用户的新问题，不再重复发送旧图谱全文",
}
compact_messages = [overloaded_messages[0], {"role": "user", "content": "COMPACT SUMMARY\n" + json.dumps(summary, ensure_ascii=False, indent=2)}]
print("before characters:", len(json.dumps(overloaded_messages, ensure_ascii=False)))
print("after characters:", len(json.dumps(compact_messages, ensure_ascii=False)))
print("after compact:")
print(json.dumps(compact_messages, ensure_ascii=False, indent=2))

### 一个提示词随轮数迭代的例子

从过去的连续论文共读任务中取出了一个真实 user、assistant、tool call、tool result 和 compact 前后结构。

![真实历史三页请求图](../docs/05-compact-triptych.png)